In [ ]:
import pytesseract
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from time import sleep
import datetime
from bs4 import BeautifulSoup
from pandas import ExcelWriter
from PIL import Image
import re
import os

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print("Running CZ CNBCZ Web Scraping Tool v.1.0")
print("***The program requires Tesseract, which path must be stated in the line 30 of this script. No longer update is required\n unless the regulator's site is considerably modified.***")

scriptfolder=os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

now=datetime.datetime.now()
filename= 'CZ CNBCZ {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

regdict= {'CZ CNBCZ 1': '1', 'CZ CNBCZ 2': '16', 'CZ CNBCZ 3': '133', 'CZ CNBCZ 4': '134', 'CZ CNBCZ 5': '37', 'CZ CNBCZ 6': '140', 
		  'CZ CNBCZ 7': '17', 'CZ CNBCZ 8': '31', 'CZ CNBCZ 9': '139', 'CZ CNBCZ 10': '53','CZ CNBCZ 11': '239', 'CZ CNBCZ 12': '248', 'CZ CNBCZ 13': '82', 
		  'CZ CNBCZ 14': '141', 'CZ CNBCZ 15': '235', 'CZ CNBCZ 16': '253', 'CZ CNBCZ 17': '223', 'CZ CNBCZ 18': '226', 'CZ CNBCZ 20': '224', 
		  'CZ CNBCZ 21': '231', 'CZ CNBCZ 22': '174', 'CZ CNBCZ 23': '184', 'CZ CNBCZ 24': '182', 'CZ CNBCZ 25': '183', 'CZ CNBCZ 26': '205', 
		  'CZ CNBCZ 27': '206', 'CZ CNBCZ 28': '118'}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
'Address_1 - Mother company': [], 'Address_2 - Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
'Phone - Mother company': [], 'Check': []}

catlist = ['Name', 'RECNTRY', 'REGCODE', 'LISTCODE', 'Entity Type', 'Personal identification number', 'Institution Name', 'Registered / permanent residence address', 'Contact Address', 'Phone', 'Fax', 'E-mail', 'Website', 'Numeric code', 'LEI', 'Type of authorization', 'Date of authorization', 'Date the decision came to legal force', 'Ownership Structure', 'Related legal ties', 'Other function(s)', 'Note', 'Head office', 'Detailed Entity Type', 'Branches and subsidiaries abroad', 'Cross-border services', 'Date of entry in the Companies register', 'Date of entry in the list', 'Depository', 'Management company', 'Cross-border marketing', 'EU management company', 'Local contact person', 'Registration number', 'Concession / Registration number']

label_sql = {'Entity Type' : 'Typology', 'Personal identification number' : 'InternalID_1', 'Registered / permanent residence address' : 'Address_1', 'Phone' : 'Phone', 'Fax' :'Fax', 'E-mail' : 'Email', 'Website' : 'Website', 'Numeric code' : 'InternalID_2', 'LEI' : 'LEI Code', 'Type of authorization' : 'RegulationType', 'Date of authorization' : 'RegulationDate', 'Head office' : 'Name - Mother Company'}


def get_captcha(file_path):
    captcha = pytesseract.image_to_string(file_path)
    print('1. OCRed text: ',captcha)
    captcha_answer = ''.join([char for char in captcha if char.isdigit()])
    print('2. Captcha: ',captcha_answer)
    return captcha_answer


chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

driver.get('https://apl.cnb.cz/apljerrsdad/JERRS.WEB15.BASIC_LISTINGS?p_lang=en')
sleep(3)
image_path = os.path.join(tempfolder, 'captcha.png')
image = driver.find_element(By.ID, 'ID_EMAIL_FORM').screenshot(image_path)
#driver.find_element(By.XPATH, '//*[@id="OPIS"]').click() #clicking on the captcha field input field
driver.find_element(By.XPATH, '//*[@id="OPIS"]').send_keys(get_captcha(image_path)) #Sending captcha answer
os.remove(image_path)
sleep(1)
sendcap = driver.find_element(By.XPATH, '//input[@value="Confirm"]').click()#Confirming captcha
sleep(2)
step0 = driver.find_element(By.LINK_TEXT, 'Predefined lists').click()
sleep(2)
step1 = driver.find_element(By.XPATH, '//*[@id="ID_REC_PER_PAGE"]/option[@value="108"]').click() # list value 108
sleep(2)
step2 = driver.find_element(By.XPATH, '//input[@value="Next step(s)"]').click()
sleep(2)
soup = BeautifulSoup(driver.page_source, 'html.parser')

#First grabbing URLs for each list, this is done because they are unique they include date

for reg in regdict:
    print(f"Grabbing URL for {reg}")
    xlink=soup.find(text=re.compile('Banks and branches of foreign banks'), href= True)
    xlink=str(xlink['href'])
    while xlink[-1].isdigit():
        xlink = xlink[:-1]
    regdict[reg] = 'https://apl.cnb.cz/apljerrsdad/'+xlink+regdict[reg]

#Loop for every list code with the correct URL
for reg in regdict:
    print(f"Working with {reg}")
    driver.get(regdict[reg])
    sleep(1)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    entities = soup.find_all("a", {"class":"textNorm"}, href= True)
    entities = [(ele.text.strip(), "https://apl.cnb.cz/apljerrsdad/"+str(ele['href'])) for ele in entities if 'DOWNLOAD' not in ele.text.upper()]
    for name, url in entities:
        driver.get(url)
        sleep(0.5)
        soup=BeautifulSoup(driver.page_source, "html.parser")
        #Verifying if we were sent to initial page
        if soup.find("p", {"id":"nadpisStranky"}).text=='Regulated institutions and registered financial market entities lists':
            driver.get('https://apl.cnb.cz/apljerrsdad/JERRS.WEB15.BASIC_LISTINGS?p_lang=en')
            sleep(2)
            image = driver.find_element(By.ID, 'ID_EMAIL_FORM').screenshot(image_path) #we use the same string for all our image file names since we delete the files after use
            driver.find_element(By.XPATH, '//*[@id="OPIS"]').send_keys(get_captcha(image_path))#Sending captcha
            os.remove(image_path)
            sendcap = driver.find_element(By.XPATH, '//input[@value="Confirm"]').click()#Confirming captcha
            driver.get(url)
            sleep(0.5)
            soup = BeautifulSoup(driver.page_source, "html.parser")
        soup = BeautifulSoup(driver.page_source.replace('<br>', ', ').replace('<br/>', ', ').replace('&nbsp;', ' '), "html.parser")
        table = soup.find('table').find('tbody')
        sqldict['Name'].append(name)
        sqldict['RegCtry'].append('CZ')
        sqldict['RegCode'].append('CNBCZ')
        sqldict['ListCode'].append(reg.split()[-1])
        sqldict['InternalID_1_type'].append('Personal identification number')
        sqldict['InternalID_2_type'].append('Numeric code')
        for tr in table.find_all('tr'):
            labels = tr.find_all('td', {'class': 'tableNadpis'})
            values = tr.find_all('td', {'class': 'tableDetail'})
            if len(labels) + len(values) == 0 or len(labels)!= len(values):
                continue
            labels = [ele.text.strip() for ele in labels]
            values = [ele.text.strip() for ele in values]
            for rang, label_ in enumerate(labels):
                if label_ in label_sql and len(sqldict[label_sql[label_]])<len(sqldict['Name']):
                    sqldict[label_sql[label_]].append(values[rang])
        for key in sqldict:
            if len(sqldict['Name'])>len(sqldict[key]):
                sqldict[key].append('')

df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    
    